# Lezione 4: Dataset, DataLoader e Protocollo Held-Out
**File**: [`data/dataset.py`](file:///g:/My%20Drive/PhD%20&%20Research/Temporal%20GNN/cfm_grn/data/dataset.py)  
**Ruolo nel progetto**: Gestisce la divisione rigorosa tra perturbazioni di addestramento e perturbazioni **Held-Out (nuove, mai viste)** per testare la generalizzazione.


---
## 1. La Sfida Scientifica: Generalizzazione Fuori Distribuzione (OOD)

Nella ricerca su CRISPR a singola cellula (es. Perturb-seq), profilare sperimentalmente tutti i ~20.000 geni umani è proibitivo per costi e tempi.
L'obiettivo fondamentale dell'AI in questo campo è:
> **Addestrare il modello su un sottoinsieme di geni perturbati e prevedere con successo l'effetto del knockout di geni TOTALMENTE NUOVI (mai visti durante il training)!**

Se dividessimo i dati facendo uno split casuale sulle cellule dello stesso knockout (es. 80% train, 20% test per il Gene A), il modello potrebbe semplicemente memorizzare l'espressione media del Gene A. Questo è un errore metodologico grave (**data leakage**).

La classe `PerturbationPairDataset` implementa il corretto **Split a Livello di Gene (Held-Out Target)**.


---
## 2. Setup e Import


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
from data.synthetic_grn import SyntheticGRNSimulator
from data.dataset import PerturbationPairDataset, create_dataloaders
print('Moduli dataset caricati con successo!')


---
## 3. Creazione del Benchmark con Gene Held-Out


In [ ]:
sim = SyntheticGRNSimulator(num_genes=12, density=0.25, seed=42)
benchmark_data = sim.generate_benchmark_dataset(cells_per_condition=300)

perturbed_genes = benchmark_data['perturbed_genes']
held_out_gene = perturbed_genes[0]  # Escludiamo il primo gene perturbato!
train_genes = perturbed_genes[1:]

print(f'Tutti i geni perturbati disponibili: {perturbed_genes}')
print(f'Geni usati per il TRAINING: {train_genes}')
print(f'Gene HELD-OUT per il TEST (escluso dal training): {held_out_gene}')


In [ ]:
# Creiamo i DataLoader separati
train_loader, test_loader = create_dataloaders(
    benchmark_data=benchmark_data,
    batch_size=64,
    held_out_genes=[held_out_gene]
)

print(f'Numero di batch di Training: {len(train_loader)}')
print(f'Numero di batch di Held-Out Test: {len(test_loader)}')


---
## 4. Ispezione di un Batch di Dati

Ogni iterazione del DataLoader restituisce una tupla di 4 elementi:
1. `x_0` $[B, G]$: Profili delle cellule di controllo (sane).
2. `x_1` $[B, G]$: Profili delle cellule perturbate.
3. `pert_vec` $[B, G]$: Vettore one-hot che indica quale gene è stato silenziato.
4. `gene_id` $[B]$: ID intero del gene perturbato.


In [ ]:
for x_0, x_1, pert_vec, gene_id in train_loader:
    print('Shape x_0 (controllo):', x_0.shape)
    print('Shape x_1 (perturbato):', x_1.shape)
    print('Shape pert_vec (one-hot):', pert_vec.shape)
    print('Geni perturbati presenti in questo batch di train:', torch.unique(gene_id).tolist())
    # Verifichiamo che il gene held-out NON compaia MAI nel training batch!
    assert held_out_gene not in gene_id.tolist(), 'ERRORE: Gene held-out trovato nel train set!'
    break

print('Verifica superata: nessuna contaminazione nel training set!')
